# Week 2 — GP + UCB (Exploration Phase) 

**What this notebook does**
- Loads Week 0 starter data from the organiser-provided `.npy` files.
- Adds **Week 1** observed (input, output) pairs.
- Fits a **Gaussian Process surrogate** per function.
- Runs a **UCB acquisition** ($\mu + \beta\sigma$, typically with a higher $\beta$ early on).
- Records the **actual Week 2 portal submissions**



In [64]:
import numpy as np
import pandas as pd

W1_DIR = 'initial_data' 

def load_week1(function_id: int):
    X = np.load(f'{W1_DIR}/function_{function_id}/F{function_id}_initial_inputs.npy')
    y = np.load(f'{W1_DIR}/function_{function_id}/F{function_id}_initial_outputs.npy')
    return X, y

for f in range(1,9):
    X0,y0 = load_week1(f)
    print(f'F{f}: X0={X0.shape}, y0={y0.shape}, y0 min/max=({y0.min():.4g}, {y0.max():.4g})')


F1: X0=(10, 2), y0=(10,), y0 min/max=(-0.003606, 7.711e-16)
F2: X0=(10, 2), y0=(10,), y0 min/max=(-0.06562, 0.6112)
F3: X0=(15, 3), y0=(15,), y0 min/max=(-0.3989, -0.03484)
F4: X0=(30, 4), y0=(30,), y0 min/max=(-32.63, -4.026)
F5: X0=(20, 4), y0=(20,), y0 min/max=(0.1129, 1089)
F6: X0=(20, 5), y0=(20,), y0 min/max=(-2.571, -0.7143)
F7: X0=(30, 6), y0=(30,), y0 min/max=(0.002701, 1.365)
F8: X0=(40, 8), y0=(40,), y0 min/max=(5.592, 9.598)


## Add Week 1 data and build the Week 2 training set

In [65]:
week1_inputs = {
  1: np.array([0.759779, 0.804108]),
  2: np.array([0.717915, 0.002064]),
  3: np.array([0.994942, 0.051967, 0.792526]),
  4: np.array([0.404477, 0.413254, 0.303108, 0.434359]),
  5: np.array([0.940282, 0.064909, 0.998637, 0.996528]),
  6: np.array([0.379776, 0.684419, 0.068171, 0.984146, 0.290503]),
  7: np.array([0.015298, 0.590564, 0.658382, 0.751493, 0.425786, 0.776978]),
  8: np.array([0.034492, 0.437681, 0.011459, 0.323393, 0.989664, 0.04578, 0.097175, 0.702465]),
}
week1_outputs = {
  1: 3.1007222612420183e-34,
  2: 0.5697925224470959,
  3: -0.15184421067439557,
  4: -0.7158150747637886,
  5: 3653.9508244023123,
  6: -1.362061899095797,
  7: 0.23953469687403112,
  8: 9.5899381138985,
}

def build_training_set_for_week2(f_id: int):
    X0,y0 = load_week1(f_id)
    x1 = week1_inputs[f_id].reshape(1,-1)
    y1 = np.array([week1_outputs[f_id]])
    X = np.vstack([X0, x1])
    y = np.concatenate([y0, y1])
    return X,y

for f in [1,2,3]:
    X,y = build_training_set_for_week2(f)
    print(f'F{f}: n_train={len(y)}, d={X.shape[1]}')


F1: n_train=11, d=2
F2: n_train=11, d=2
F3: n_train=16, d=3


## GP + UCB implementation (exploration-friendly)

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.preprocessing import StandardScaler

def fit_gp(X, y, noise=1e-6, seed=42, n_restarts=5):
    ys = StandardScaler()
    y_std = ys.fit_transform(y.reshape(-1,1)).ravel()
    kernel = C(1.0,(1e-3,1e3)) * Matern(length_scale=np.ones(X.shape[1]), nu=2.5) + WhiteKernel(noise_level=noise)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=n_restarts, random_state=seed)
    gp.fit(X, y_std)
    return gp

def propose_ucb(gp, d, beta=2.0, n_candidates=200_000, seed=123):
    rng = np.random.default_rng(seed)
    Cands = rng.uniform(0.0, 1.0, size=(n_candidates, d))
    mu, std = gp.predict(Cands, return_std=True)
    ucb = mu + beta*std
    return Cands[int(np.argmax(ucb))]

def fmt_portal(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


## Run UCB 

In [ ]:
beta = 2.0
n_candidates = 500_000

ucb_recos = {}
for f in range(1,9):
    X,y = build_training_set_for_week2(f)
    gp = fit_gp(X,y,seed=200+f)
    x2 = propose_ucb(gp, d=X.shape[1], beta=beta, n_candidates=n_candidates, seed=100+f)
    ucb_recos[f] = x2
    print(f'F{f}:', fmt_portal(x2))


##  Week 2 submissions 

In [66]:
week2_inputs = {
  1: np.array([0.803235, 0.719438]),
  2: np.array([0.957387, 0.923033]),
  3: np.array([0.009308, 0.062825, 0.001028]),
  4: np.array([0.398003, 0.395708, 0.361254, 0.442257]),
  5: np.array([0.99741, 0.041752, 0.791796, 0.974979]),
  6: np.array([0.493449, 0.127064, 0.631478, 0.904964, 0.008065]),
  7: np.array([0.015116, 0.182436, 0.288522, 0.173288, 0.397265, 0.665194]),
  8: np.array([0.358728, 0.042807, 0.355074, 0.019341, 0.809116, 0.05435, 0.053363, 0.000727]),
}

for f in range(1,9):
    print(f'Function {f}:', fmt_portal(week2_inputs[f]))


Function 1: 0.803235-0.719438
Function 2: 0.957387-0.923033
Function 3: 0.009308-0.062825-0.001028
Function 4: 0.398003-0.395708-0.361254-0.442257
Function 5: 0.997410-0.041752-0.791796-0.974979
Function 6: 0.493449-0.127064-0.631478-0.904964-0.008065
Function 7: 0.015116-0.182436-0.288522-0.173288-0.397265-0.665194
Function 8: 0.358728-0.042807-0.355074-0.019341-0.809116-0.054350-0.053363-0.000727
